In [2]:
pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [1]:

from itertools import product
import openpyxl
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry


def build_dhis2_metadata_map(
    session: requests.Session, base_url: str
) -> dict[str, str]:
    """Queries DHIS2 API to construct a UID-to-DisplayName lookup dictionary

    for Attributes, Data Elements, and Program Stages.
    """
    meta_map = {}

    endpoints = {
        "trackedEntityAttributes": f"{base_url}/api/trackedEntityAttributes.json?fields=id,displayName&paging=false",
        "dataElements": f"{base_url}/api/dataElements.json?fields=id,displayName&paging=false",
        "programStages": f"{base_url}/api/programStages.json?fields=id,displayName&paging=false",
        "programs": f"{base_url}/api/programs.json?fields=id,displayName&paging=false",
        "organisationUnits": f"{base_url}/api/organisationUnits.json?fields=id,displayName&paging=false",
    }

    print("Fetching DHIS2 metadata display names...")
    for resource, url in endpoints.items():
        try:
            res = session.get(url, timeout=30)
            if res.ok:
                items = res.json().get(resource, [])
                for item in items:
                    meta_map[item["id"]] = item.get("displayName", item["id"])
        except Exception as e:
            print(f"Warning: Failed to fetch {resource} metadata: {e}")

    print(f"Loaded {len(meta_map)} metadata display name mappings.")
    return meta_map


def get_data_dhis2(
    web: str,
    username: str,
    password: str,
    idprogram: list[str] | str,
    idou: list[str] | str,
) -> pd.DataFrame:
    """Extracts tracked entity instances for multiple programs and org units,

    resolving all attribute, data element, and stage IDs into human-readable Display Names.
    """
    programs = [idprogram] if isinstance(idprogram, str) else idprogram
    org_units = [idou] if isinstance(idou, str) else idou

    base_url = web.rsplit("/api/", 1)[0].rstrip("/")
    endpoint = f"{base_url}/api/trackedEntityInstances.json"

    # Setup Session with retries
    session = requests.Session()
    session.auth = (username, password)
    session.headers.update(
        {
            "Accept": "application/json",
            "User-Agent": "DHIS2-Python-Script/1.0",
        }
    )

    retries = Retry(
        total=5, backoff_factor=2, status_forcelist=[500, 502, 503, 504]
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    # 1. Fetch Metadata Map for Display Names
    meta_map = build_dhis2_metadata_map(session, base_url)

    all_records = []

    # 2. Extract Data
    for prog, ou in product(programs, org_units):
        page = 1
        page_size = 100

        while True:
            params = {
                "program": prog,
                "ou": ou,
                "ouMode": "DESCENDANTS",
                "pageSize": page_size,
                "page": page,
                "totalPages": "true",
                "fields": "trackedEntityInstance,orgUnit,created,lastUpdated,attributes[attribute,displayName,value],enrollments[enrollment,program,orgUnit,enrolledAt,occurredAt,status,events[event,programStage,occurredAt,status,dataValues[dataElement,value]]]",
            }

            try:
                response = session.get(endpoint, params=params, timeout=45)
                if not response.ok:
                    print(
                        f"DHIS2 Error (Prog: {prog}, OU: {ou}, Status {response.status_code}):"
                    )
                    print(response.text)
                    response.raise_for_status()

                data = response.json()
                instances = data.get("trackedEntityInstances", [])

                if not instances:
                    break

                for instance in instances:
                    # Resolve OrgUnit display name
                    ou_id = instance.get("orgUnit")
                    ou_name = meta_map.get(ou_id, ou_id)

                    # Resolve Program display name
                    prog_name = meta_map.get(prog, prog)

                    row = {
                        "program_id": prog,
                        "program_name": prog_name,
                        "trackedEntityInstance": instance.get(
                            "trackedEntityInstance"
                        ),
                        "orgUnit_id": ou_id,
                        "Org Unit Name": ou_name,
                        "created": instance.get("created"),
                        "lastUpdated": instance.get("lastUpdated"),
                    }

                    # A. Map Patient/Entity Attributes using displayName with Fallback
                    for attr in instance.get("attributes", []):
                        attr_id = attr.get("attribute")
                        # Priority: 1. API displayName -> 2. Metadata Map -> 3. Attribute UID
                        col_name = (
                            attr.get("displayName")
                            or meta_map.get(attr_id)
                            or attr_id
                        )
                        row[col_name] = attr.get("value")

                    # B. Map Program Stage Events & Data Values
                    for enrollment in instance.get("enrollments", []):
                        if enrollment.get("program") == prog:
                            row["enrollment_date"] = enrollment.get(
                                "enrolledAt"
                            )
                            row["enrollment_status"] = enrollment.get("status")

                            for event in enrollment.get("events", []):
                                stage_id = event.get("programStage")
                                stage_name = meta_map.get(
                                    stage_id, f"Stage_{stage_id}"
                                )

                                for dv in event.get("dataValues", []):
                                    de_id = dv.get("dataElement")
                                    de_name = meta_map.get(
                                        de_id, f"Element_{de_id}"
                                    )

                                    # Prefix Data Element with Stage Name to prevent collisions across multiple stages
                                    col_key = f"[{stage_name}] {de_name}"
                                    row[col_key] = dv.get("value")

                    all_records.append(row)

                # Pagination
                pager = data.get("pager", {})
                page_count = pager.get("pageCount", 1)

                if page >= page_count:
                    break

                page += 1

            except requests.exceptions.RequestException as e:
                print(f"Failed fetching Prog: {prog}, OU: {ou}, Page {page}: {e}")
                break

    session.close()
    return pd.DataFrame(all_records)


def export_dhis2_to_excel(
    df: pd.DataFrame,
    output_filename: str = "DHIS2_Tracker_MultiProgram_Export.xlsx",
):
    """Exports DataFrame into separate styled worksheets named after Program Display Names."""
    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    header_fill = PatternFill(
        start_color="1F4E79", end_color="1F4E79", fill_type="solid"
    )
    header_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
    data_font = Font(name="Segoe UI", size=10)
    alt_fill = PatternFill(
        start_color="F2F5F9", end_color="F2F5F9", fill_type="solid"
    )
    thin_border = Border(
        left=Side(style="thin", color="D9D9D9"),
        right=Side(style="thin", color="D9D9D9"),
        top=Side(style="thin", color="D9D9D9"),
        bottom=Side(style="thin", color="D9D9D9"),
    )

    if df.empty:
        ws = wb.create_sheet(title="Data")
        ws.views.sheetView[0].showGridLines = True
        ws.append(["No records found."])
        wb.save(output_filename)
        return

    # Group by program_name or program_id
    group_col = "program_name" if "program_name" in df.columns else "program_id"
    grouped = df.groupby(group_col, dropna=False)

    for prog_label, prog_df in grouped:
        # Clean worksheet name (max 31 chars)
        sheet_title = (
            str(prog_label)[:30] if pd.notna(prog_label) else "Unknown_Program"
        )
        ws = wb.create_sheet(title=sheet_title)
        ws.views.sheetView[0].showGridLines = True

        # Drop columns with all NaN for this program tab
        prog_df_clean = prog_df.dropna(how="all", axis=1)

        headers = list(prog_df_clean.columns)
        ws.append(headers)

        for col_idx in range(1, len(headers) + 1):
            cell = ws.cell(row=1, column=col_idx)
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(
                horizontal="center", vertical="center", wrap_text=True
            )

        records = prog_df_clean.to_dict(orient="records")
        for row_idx, record in enumerate(records, start=2):
            row_values = [record.get(col, "") for col in headers]
            ws.append(row_values)

            is_even = row_idx % 2 == 0
            for col_idx in range(1, len(headers) + 1):
                cell = ws.cell(row=row_idx, column=col_idx)
                cell.font = data_font
                cell.border = thin_border
                cell.alignment = Alignment(vertical="center")
                if is_even:
                    cell.fill = alt_fill

        ws.freeze_panes = "A2"

        for col in ws.columns:
            max_len = max(len(str(cell.value or "")) for cell in col)
            col_letter = get_column_letter(col[0].column)
            ws.column_dimensions[col_letter].width = min(
                max(max_len + 4, 12), 45
            )

    wb.save(output_filename)
    print(f"Saved Excel file with resolved Display Names: {output_filename}")


# --- Execution ---
WEB_URL = "https://hmistraining.mm.dhis2.net/train"
USERNAME = "MATA_ntp"
PASSWORD = "District@1"

PROGRAM_IDS = [
    "UZF0HrTlps0",
    "GvywHD6crky",
    "Lt6P15ps7f6",
    "cQsXTtAJ3HW",
]

ORG_UNIT_IDS = [
    "ZL8xjYM5wGV",
    "KqjORlUe8Yc",
]

df = get_data_dhis2(WEB_URL, USERNAME, PASSWORD, PROGRAM_IDS, ORG_UNIT_IDS)
export_dhis2_to_excel(df, "DHIS2_Tracker_DisplayName_Export.xlsx")



Fetching DHIS2 metadata display names...
Loaded 20160 metadata display name mappings.
Saved Excel file with resolved Display Names: DHIS2_Tracker_DisplayName_Export.xlsx


In [ ]:
#### All fields within raw data

from itertools import product
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry


def get_data_dhis2(
    web: str,
    username: str,
    password: str,
    idprogram: list[str] | str,
    idou: list[str] | str,
) -> pd.DataFrame:
    """Extracts tracked entity instances, program attributes, and stage data values

    for multiple DHIS2 programs and org units into a flattened DataFrame.
    """
    programs = [idprogram] if isinstance(idprogram, str) else idprogram
    org_units = [idou] if isinstance(idou, str) else idou

    base_url = web.rsplit("/api/", 1)[0].rstrip("/")
    endpoint = f"{base_url}/api/trackedEntityInstances.json"

    # Setup resilient HTTP session
    session = requests.Session()
    session.auth = (username, password)
    session.headers.update(
        {
            "Accept": "application/json",
            "User-Agent": "DHIS2-Python-Script/1.0",
        }
    )

    retries = Retry(
        total=5, backoff_factor=2, status_forcelist=[500, 502, 503, 504]
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    all_records = []

    for prog, ou in product(programs, org_units):
        page = 1
        page_size = 100

        while True:
            params = {
                "program": prog,
                "ou": ou,
                "ouMode": "DESCENDANTS",
                "pageSize": page_size,
                "page": page,
                "totalPages": "true",
                # Expand fields parameter to capture all attributes, enrollments, events & data values
                "fields": "trackedEntityInstance,orgUnit,created,lastUpdated,inactive,deleted,attributes[attribute,displayName,value],enrollments[enrollment,program,orgUnit,enrolledAt,occurredAt,status,events[event,programStage,occurredAt,status,dataValues[dataElement,value]]]",
            }

            try:
                response = session.get(endpoint, params=params, timeout=45)
                if not response.ok:
                    print(
                        f"DHIS2 Error (Prog: {prog}, OU: {ou}, Status {response.status_code}):"
                    )
                    print(response.text)
                    response.raise_for_status()

                data = response.json()
                instances = data.get("trackedEntityInstances", [])

                if not instances:
                    break

                for instance in instances:
                    row = {
                        "program_id": prog,
                        "trackedEntityInstance": instance.get(
                            "trackedEntityInstance"
                        ),
                        "orgUnit": instance.get("orgUnit"),
                        "created": instance.get("created"),
                        "lastUpdated": instance.get("lastUpdated"),
                        "inactive": instance.get("inactive", False),
                    }

                    # 1. Extract Profile / Patient Attributes
                    for attr in instance.get("attributes", []):
                        col_name = (
                            attr.get("displayName") or attr.get("attribute")
                        )
                        row[col_name] = attr.get("value")

                    # 2. Extract Enrollments, Program Stages, and Data Values
                    for enrollment in instance.get("enrollments", []):
                        # Filter to process enrollments matching the current queried program
                        if enrollment.get("program") == prog:
                            row["enrollment_id"] = enrollment.get("enrollment")
                            row["enrollment_date"] = enrollment.get(
                                "enrolledAt"
                            )
                            row["enrollment_status"] = enrollment.get("status")

                            # Extract stage events and clinical data values
                            for event in enrollment.get("events", []):
                                stage_id = event.get("programStage")
                                for dv in event.get("dataValues", []):
                                    de_id = dv.get("dataElement")
                                    # Scope data elements by program stage to prevent overwrite collisions
                                    col_key = f"{stage_id}_{de_id}"
                                    row[col_key] = dv.get("value")

                    all_records.append(row)

                # Pagination check
                pager = data.get("pager", {})
                page_count = pager.get("pageCount", 1)

                if page >= page_count:
                    break

                page += 1

            except requests.exceptions.RequestException as e:
                print(f"Failed fetching Prog: {prog}, OU: {ou}, Page {page}: {e}")
                break

    session.close()
    return pd.DataFrame(all_records)


# --- Run Script ---
WEB_URL = "https://hmistraining.mm.dhis2.net/train"
USERNAME = "MATA_ntp"
PASSWORD = "District@1"

PROGRAM_IDS = [
    "UZF0HrTlps0",  # MOD1
    "GvywHD6crky",  # MOD2
    "Lt6P15ps7f6",  # MOD3
    "cQsXTtAJ3HW",  # MOD4
]

ORG_UNIT_IDS = [
    "ZL8xjYM5wGV",  # NTP
    "KqjORlUe8Yc",  # MATA
]

df = get_data_dhis2(WEB_URL, USERNAME, PASSWORD, PROGRAM_IDS, ORG_UNIT_IDS)
print(
    f"Successfully retrieved {len(df)} total rows with {len(df.columns)} total fields."
)
